# **Calculando $\mathcal{R}_0$ con `pyR0compute`**

En este script se presentan varios ejemplos del cálculo simbólico del ***número reproductivo básico*** mediante el método de la **matriz de próxima generación** (van den Driessche y Watmough, 2002), en el que $\mathcal{R}_0 = \rho(FV^{-1})$. 

El usuario solo debe escribir el **sistema de EDO** e indica los **compartimentos infectados**. La librería reconoce automáticamente como parámetros todos los símbolos que no son variables de estado ni expresiones auxiliares, calcula el equilibrio libre de enfermedad y separa los términos de nuevas infecciones ($\mathcal{F}$) de los de transición ($\mathcal{V}$). Como esa separación no es única, la clasificación automática puede reemplazarse con `new_infections={...}`, y el equilibrio, cuando no queda determinado por las ecuaciones, puede fijarse con `dfe={...}`.

Los ejemplos abarcan modelos epidemiológicos clásicos, transmisión vectorial, dinámica intrahuésped y competencia entre cepas. Además, se calculan índices de sensibilidad normalizados y se generan reportes del procedimiento en código $\LaTeX$, listos para copiar en un documento.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/01_ejemplos.ipynb)

In [1]:
%pip install pyR0compute                      # Instala la librería pyR0compute desde PyPI 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sympy as sp                            # Importa SymPy para realizar cálculo simbólico
from pyr0compute import R0Model               # Importa la clase R0Model desde el paquete pyr0compute
from IPython.display import Markdown, display # Importa Markdown y display para mostrar resultados en formato Markdown en Jupyter Notebook

## 1. Modelo SEIR

$$
\begin{cases}
\dfrac{dS}{dt} = \Lambda - \beta SI - \mu S \\[6pt]
\dfrac{dE}{dt} = \beta SI - (\sigma + \mu)E \\[6pt]
\dfrac{dI}{dt} = \sigma E - (\gamma + \mu)I \\[6pt]
\dfrac{dR}{dt} = \gamma I - \mu R
\end{cases}
$$

Solo se entregan las ecuaciones y los compartimentos infectados.

In [3]:
seir = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["E", "I"])

seir.R0

Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma))

In [4]:
print("Parámetros detectados:", seir.parameters)
print("Equilibrio libre de enfermedad:", seir.dfe)

Parámetros detectados: (Lambda, beta, gamma, mu, sigma)
Equilibrio libre de enfermedad: {S: Lambda/mu, E: 0, I: 0, R: 0}


Las matrices $F$, $V$ y $K = FV^{-1}$ están disponibles como objetos de SymPy:

In [5]:
seir.F, seir.V, seir.K

(Matrix([
 [0, Lambda*beta/mu],
 [0,              0]]),
 Matrix([
 [mu + sigma,          0],
 [    -sigma, gamma + mu]]),
 Matrix([
 [Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma)), Lambda*beta/(mu*(gamma + mu))],
 [                                               0,                             0]]))

In [6]:
display(Markdown(seir.report_latex("markdown")))

## pyR0compute: next-generation matrix method

**Variables:** $S, E, I, R$  
**Infected compartments:** $E, I$  
**Parameters (detected automatically):** $\Lambda, \beta, \gamma, \mu, \sigma$

### Model

$$
\begin{aligned}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{aligned}
$$

### New-infection terms (automatic)

$$
\begin{aligned}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{aligned}
$$

### Term classification

| Equation | Term | Class | Reason |
|---|---|:---:|---|
| $dE/dt$ | $E \left(- \mu - \sigma\right)$ | $\mathcal{V}$ | negative term |
| $dE/dt$ | $I S \beta$ | $\mathcal{F}$ | transfer from uninfected S |
| $dI/dt$ | $E \sigma$ | $\mathcal{V}$ | transfer from infected E |
| $dI/dt$ | $I \left(- \gamma - \mu\right)$ | $\mathcal{V}$ | negative term |

### Transition terms

$$
\begin{aligned}
\mathcal{V}_{E} &= E \mu + E \sigma \\
\mathcal{V}_{I} &= - E \sigma + I \gamma + I \mu
\end{aligned}
$$

### Disease-free equilibrium

$$
\begin{aligned}
S &= \frac{\Lambda}{\mu} \\
E &= 0 \\
I &= 0 \\
R &= 0
\end{aligned}
$$

### Next-generation matrix

$$
\begin{aligned}
F &= \begin{bmatrix}0 & \frac{\Lambda \beta}{\mu}\\0 & 0\end{bmatrix} \\
V &= \begin{bmatrix}\mu + \sigma & 0\\- \sigma & \gamma + \mu\end{bmatrix} \\
K = F V^{-1} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)} & \frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}\\0 & 0\end{bmatrix}
\end{aligned}
$$

Only $E$ receive new infections, so $\mathcal{R}_0$ is the spectral radius of the next-generation matrix with small domain:

$$
\begin{aligned}
K_{\text{small}} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}\end{bmatrix}
\end{aligned}
$$

### Basic reproduction number

$$
\begin{aligned}
\mathcal{R}_0 = \rho\left(F V^{-1}\right) &= \frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}
\end{aligned}
$$

In [7]:
print(seir.report_latex())

\section*{pyR0compute: next-generation matrix method}

\textbf{Variables:} $S, E, I, R$ \\
\textbf{Infected compartments:} $E, I$ \\
\textbf{Parameters (detected automatically):} $\Lambda, \beta, \gamma, \mu, \sigma$

\subsection*{Model}

\begin{align*}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{align*}

\subsection*{New-infection terms (automatic)}

\begin{align*}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{align*}

\subsection*{Term classification}

\begin{tabular}{llcl}
\hline
Equation & Term & Class & Reason \\
\hline
$dE/dt$ & $E \left(- \mu - \sigma\right)$ & $\mathcal{V}$ & negative term \\
$dE/dt$ & $I S \beta$ & $\mathcal{F}$ & transfer from uninfected S \\
$dI/dt$ & $E \sigma$ & $\mathcal{V}$ & transfer from infected E \\
$dI/dt$ & $I \left(- \gamma - \mu\right)$ & $\mathcal{V}$ & negative term \\


In [8]:
print(seir.latex())

\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}


### Índices de sensibilidad

$\Upsilon_p = \dfrac{\partial R_0}{\partial p}\dfrac{p}{R_0}$: un aumento de 1% en $p$ modifica $R_0$ en $\Upsilon_p$%.

In [9]:
seir.sensitivity_indices()

{Lambda: 1,
 beta: 1,
 gamma: -gamma/(gamma + mu),
 mu: -(2*gamma*mu + gamma*sigma + 3*mu**2 + 2*mu*sigma)/((gamma + mu)*(mu + sigma)),
 sigma: mu/(mu + sigma)}

In [10]:
values = {"Lambda": 10, "beta": 0.001, "sigma": 0.2, "gamma": 0.1, "mu": 0.02}
print("R0 =", seir.R0_numeric(values))
seir.sensitivity_indices(values)

R0 = 3.787878787878788


{Lambda: 1.0,
 beta: 1.0,
 gamma: -0.8333333333333333,
 mu: -1.2575757575757578,
 sigma: 0.09090909090909093}

## **2. Modelo SIR clásico (población cerrada)**

$$
\begin{cases}
\dfrac{dS}{dt}
= -\dfrac{\beta S I}{N} \\[8pt]
\dfrac{dI}{dt}
= \dfrac{\beta S I}{N} - \gamma I \\[8pt]
\dfrac{dR}{dt}
= \gamma I
\end{cases}
$$

### Condiciones del modelo

- **Compartimento infectado:** $I$.
- **Equilibrio libre de enfermedad (DFE):**
  $$S = N,\qquad I = 0,\qquad R = 0.$$

In [11]:
sir = R0Model({"S": "-beta*S*I/N",
               "I": "beta*S*I/N - gamma*I",
               "R": "gamma*I"}, infected=["I"], dfe={"S": "N"})
sir.R0

beta/gamma

Sin `dfe`, el mensaje de error indica exactamente qué falta:

In [12]:
try:
    R0Model({"S": "-beta*S*I/N", 
             "I": "beta*S*I/N - gamma*I", 
             "R": "gamma*I"}, infected=["I"])
except Exception as e:
    print(type(e).__name__, "->", e)

DiseaseFreeEquilibriumError -> The disease-free value of ['S'] is not determined by the equations (e.g. a closed population without births). Provide it with dfe={'S': ...}.


## **3. Enfermedad transmitida por vectores: modelo de Ross-Macdonald**

$$
\begin{cases}
\dfrac{dS_h}{dt}
= L_h - \dfrac{a b_h S_h I_v}{N_h} - \mu_h S_h \\[8pt]
\dfrac{dI_h}{dt}
= \dfrac{a b_h S_h I_v}{N_h}
- (\gamma + \mu_h) I_h \\[8pt]
\dfrac{dR_h}{dt}
= \gamma I_h - \mu_h R_h \\[8pt]
\dfrac{dS_v}{dt}
= L_v - \dfrac{a b_v S_v I_h}{N_h} - \mu_v S_v \\[8pt]
\dfrac{dI_v}{dt}
= \dfrac{a b_v S_v I_h}{N_h} - \mu_v I_v
\end{cases}
$$

### Variables de estado

- $S_h$: número de **huéspedes susceptibles**.
- $I_h$: número de **huéspedes infectados**.
- $R_h$: número de **huéspedes recuperados**.
- $S_v$: número de **vectores susceptibles**.
- $I_v$: número de **vectores infectados**.
- $N_h = S_h + I_h + R_h$: población total de huéspedes.


In [13]:
rm = R0Model("""
    dS_h/dt = L_h - a*b_h*S_h*I_v/N_h - mu_h*S_h
    dI_h/dt = a*b_h*S_h*I_v/N_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h
    dS_v/dt = L_v - a*b_v*S_v*I_h/N_h - mu_v*S_v
    dI_v/dt = a*b_v*S_v*I_h/N_h - mu_v*I_v
""", infected=["I_h", "I_v"])
rm.new_infections

{I_h: I_v*S_h*a*b_h/N_h, I_v: I_h*S_v*a*b_v/N_h}

In [14]:
rm.K

Matrix([
[                                  0, L_h*a*b_h/(N_h*mu_h*mu_v)],
[L_v*a*b_v/(N_h*mu_v*(gamma + mu_h)),                         0]])

In [15]:
rm.R0

sqrt(L_h)*sqrt(L_v)*a*sqrt(b_h)*sqrt(b_v)/(N_h*sqrt(mu_h)*mu_v*sqrt(gamma + mu_h))

## **4. Modelo huésped-vector con transmisión humano-humano y vectores con crecimiento logístico**

$$
\begin{cases}
N_h = S_h + E_h + I_h + R_h \\[6pt]
\dfrac{dS_h}{dt}
= \Delta_h
- \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
- \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\theta + \mu_h)S_h \\[8pt]
\dfrac{dE_h}{dt}
= \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
+ \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\sigma + \mu_h)E_h \\[8pt]
\dfrac{dI_h}{dt}
= \sigma E_h
- (\gamma + \mu_h)I_h \\[8pt]
\dfrac{dR_h}{dt}
= \gamma I_h
- \mu_h R_h
+ \theta S_h \\[8pt]
\dfrac{dM_v}{dt}
= \phi_v M_v
\left(1-\dfrac{M_v}{\kappa_v}\right)
- (\omega+\mu_m)M_v \\[8pt]
\dfrac{dS_v}{dt}
= \omega M_v
- \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\mu_v+\zeta)S_v \\[8pt]
\dfrac{dE_v}{dt}
= \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\psi+\mu_v+\zeta)E_v \\[8pt]
\dfrac{dI_v}{dt}
= \psi E_v
- (\mu_v+\zeta)I_v
\end{cases}
$$

### Variables de estado

#### Huéspedes humanos

- $S_h$: número de **huéspedes humanos susceptibles**.
- $E_h$: número de **huéspedes humanos expuestos**, infectados pero aún no infecciosos.
- $I_h$: número de **huéspedes humanos infectados e infecciosos**.
- $R_h$: número de **huéspedes humanos recuperados**.
- $N_h = S_h + E_h + I_h + R_h$: **población humana total**.

#### Vectores

- $M_v$: población de **vectores inmaduros**.
- $S_v$: número de **vectores adultos susceptibles**.
- $E_v$: número de **vectores adultos expuestos**, infectados pero aún no infecciosos.
- $I_v$: número de **vectores adultos infectados e infecciosos**. 

La población humana total se define como expresión auxiliar. El modelo tiene dos equilibrios libres de enfermedad (con y sin vectores); se elige el que tiene vectores y ambos quedan disponibles en `hv.dfe_candidates`.

In [16]:
hv = R0Model("""
    N_h = S_h + E_h + I_h + R_h
    dS_h/dt = Delta_h - delta*alpha_hv*S_h*I_v/N_h - beta*alpha_hh*S_h*I_h/N_h - (theta + mu_h)*S_h
    dE_h/dt = delta*alpha_hv*S_h*I_v/N_h + beta*alpha_hh*S_h*I_h/N_h - (sigma + mu_h)*E_h
    dI_h/dt = sigma*E_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h + theta*S_h
    dM_v/dt = phi_v*M_v*(1 - M_v/kappa_v) - (omega + mu_m)*M_v
    dS_v/dt = omega*M_v - delta*alpha_vh*S_v*I_h/N_h - (mu_v + zeta)*S_v
    dE_v/dt = delta*alpha_vh*S_v*I_h/N_h - (psi + mu_v + zeta)*E_v
    dI_v/dt = psi*E_v - (mu_v + zeta)*I_v
""", infected=["E_h", "I_h", "E_v", "I_v"])
hv.dfe

/tmp/ipykernel_670/135047437.py:1: UserWarning: 2 disease-free equilibria were found; using the one with the most non-zero compartments: {'S_h': Delta_h/(mu_h + theta), 'R_h': Delta_h*theta/(mu_h*(mu_h + theta)), 'M_v': -kappa_v*(mu_m + omega - phi_v)/phi_v, 'S_v': -kappa_v*omega*(mu_m + omega - phi_v)/(phi_v*(mu_v + zeta))}. Pass dfe={...} to choose another one (all are in model.dfe_candidates).
  hv = R0Model("""
/tmp/ipykernel_670/135047437.py:1: UserWarning: Entry F[2,1] = -alpha_vh*delta*kappa_v*mu_h*omega*(mu_m + omega - phi_v)/(Delta_h*phi_v*(mu_v + zeta)) is negative; new-infection terms should be non-negative.
  hv = R0Model("""


{S_h: Delta_h/(mu_h + theta),
 E_h: 0,
 I_h: 0,
 R_h: Delta_h*theta/(mu_h*(mu_h + theta)),
 M_v: -kappa_v*(mu_m + omega - phi_v)/phi_v,
 S_v: -kappa_v*omega*(mu_m + omega - phi_v)/(phi_v*(mu_v + zeta)),
 E_v: 0,
 I_v: 0}

Solo $E_h$ y $E_v$ reciben nuevas infecciones, por lo que $R_0$ es el radio espectral de la matriz de próxima generación de dominio pequeño, de tamaño $2\times 2$:

In [17]:
hv.next_generation_matrix_small

Matrix([
[                                                    alpha_hh*beta*mu_h*sigma/((gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)), alpha_hv*delta*mu_h*psi/((mu_h + theta)*(mu_v + zeta)*(mu_v + psi + zeta))],
[-alpha_vh*delta*kappa_v*mu_h*omega*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_v + zeta)),                                                                          0]])

In [18]:
hv.R0

alpha_hh*beta*mu_h*sigma/(2*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)) + sqrt(alpha_hh**2*beta**2*mu_h**2*sigma**2/((gamma + mu_h)**2*(mu_h + sigma)**2*(mu_h + theta)**2) - 4*alpha_hv*alpha_vh*delta**2*kappa_v*mu_h**2*omega*psi*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)*(mu_v + zeta)**2*(mu_v + psi + zeta)))/2

## **5. Modelo intrahuésped: células blanco, células infectadas y virus**

El modelo describe la dinámica de una infección viral dentro del huésped, considerando tres poblaciones: células blanco susceptibles a la infección, células infectadas y partículas virales libres.

$$
\begin{cases}
\dfrac{dT}{dt}
= s-dT-\beta TV \\[8pt]
\dfrac{dI}{dt}
= \beta TV-\delta I \\[8pt]
\dfrac{dV}{dt}
= pI-cV
\end{cases}
$$

### Variables de estado

* $T$: número o concentración de **células blanco susceptibles** a la infección.
* $I$: número o concentración de **células infectadas**.
* $V$: número o concentración de **partículas virales libres**.


In [19]:
tiv = R0Model("""
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", infected=["I", "V"])
tiv.R0

beta*p*s/(c*d*delta)

## **6. van den Driessche y Watmough (2002), modelo con tratamiento (§4.1)**

$$
\begin{cases}
N = S + E + I + T \\[6pt]
\dfrac{dE}{dt}
= \dfrac{\beta_1 S I}{N}
+ \dfrac{\beta_2 T I}{N}
- (d+\nu+r_1)E
+ p r_2 I \\[8pt]
\dfrac{dI}{dt}
= \nu E
- (d+r_2)I \\[8pt]
\dfrac{dS}{dt}
= b-dS
- \dfrac{\beta_1 S I}{N} \\[8pt]
\dfrac{dT}{dt}
= -dT
+ r_1E
+ q r_2I
- \dfrac{\beta_2 T I}{N}
\end{cases}
$$

### Variables de estado

- $S$: número de **individuos susceptibles**.
- $E$: número de **individuos expuestos**, infectados pero aún no infecciosos.
- $I$: número de **individuos infectados e infecciosos**.
- $T$: número de **individuos bajo tratamiento**.
- $N = S+E+I+T$: **población total**. 

In [20]:
tb = R0Model("""
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
""", infected=["E", "I"])
tb.R0

-beta1*nu/(-d**2 - d*nu - d*r1 - d*r2 + nu*p*r2 - nu*r2 - r1*r2)

## **7. Dos cepas con superinfección**

El modelo considera una población susceptible y dos cepas infecciosas que compiten entre sí. La interacción entre las cepas se representa mediante un término de **superinfección** $\nu I_1 I_2$. La cepa dominante depende de los parámetros epidemiológicos, por lo que el número reproductivo básico del sistema está determinado por el mayor número reproductivo de las dos cepas.

$$
\begin{cases}
\dfrac{dI_1}{dt}
= \beta_1 I_1 S
- (b+\gamma_1)I_1
+ \nu I_1 I_2 \\[8pt]
\dfrac{dI_2}{dt}
= \beta_2 I_2 S
- (b+\gamma_2)I_2
- \nu I_1 I_2 \\[8pt]
\dfrac{dS}{dt}
= b-bS
+\gamma_1 I_1
+\gamma_2 I_2
-(\beta_1I_1+\beta_2I_2)S
\end{cases}
$$

### Variables de estado

- $S$: proporción o número de **individuos susceptibles**.
- $I_1$: proporción o número de individuos **infectados por la cepa 1**.
- $I_2$: proporción o número de individuos **infectados por la cepa 2**.

La cepa con el mayor número reproductivo tiene ventaja epidemiológica y, bajo las condiciones del modelo, determina cuál de las dos cepas puede establecerse en la población. 

In [21]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    two = R0Model("""
        dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
        dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
        dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
    """, infected=["I1", "I2"])
    R0 = two.R0
R0

Max(beta1/(b + gamma1), beta2/(b + gamma2))

## **8. DESAFÍO: Calcula el $R_0$ de tu propio modelo**

Ahora aplica el método a un **modelo epidemiológico de tu elección**. Reemplaza las ecuaciones del ejemplo por las ecuaciones de tu modelo y especifica los compartimentos que consideras infectados.

Si `R0Model` no identifica correctamente los términos de **nuevas infecciones** y **transiciones epidemiológicas**, puedes definirlos explícitamente mediante `new_infections={...}`. Asimismo, si necesitas especificar manualmente el **equilibrio libre de enfermedad (DFE)**, puedes utilizar `dfe={...}`.

In [22]:
# Ejemplo para empezar: reemplaza estas ecuaciones por las de tu modelo
# y "infected" por los compartimentos que consideres infectados.
model = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["I"])
model.R0

Lambda*beta/(mu*(gamma + mu))

In [23]:
print(model.latex())

\frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}


# **Ejemplo con new_infections**

In [24]:
ecuaciones = """
    dS/dt = b*(S + R) + b*(1 - p)*I - beta*S*I/N - b*S
    dI/dt = beta*S*I/N + p*b*I - (gamma + b)*I
    dR/dt = gamma*I - b*R
"""

# (a) Clasificación automática
auto = R0Model(ecuaciones, infected=["I"], dfe={"S": "N"})
print("F automático:", auto.new_infections)
auto.R0

F automático: {I: I*S*beta/N}


-beta/(b*p - b - gamma)

In [25]:
# (b) Transmisión vertical tratada como nueva infección
vert = R0Model(ecuaciones, infected=["I"], dfe={"S": "N"},
               new_infections={"I": "beta*S*I/N + p*b*I"})
print("F manual:", vert.new_infections)
vert.R0

F manual: {I: I*b*p + I*S*beta/N}


(b*p + beta)/(b + gamma)

In [26]:
import sympy as sp
from pyr0compute import R0Model

modelo = R0Model("""
    dC/dt   = Lambda_C - alpha*C - mu_C*C
    dM_U/dt = alpha*C - epsilon*nu*B_E*M_U - mu_MU*M_U
    dM_I/dt = epsilon*nu*B_E*M_U - lambda*M_I*C - rho*M_I - mu_MI*M_I
    dB_E/dt = r*rho*M_I - eta*B_E*C - mu_BE*B_E
    dB_I/dt = epsilon*B_E*M_U + mu*sigma*M_I*B_I/(K_m + sigma*M_I) - theta*B_I*M_U - mu_BI*B_I
""", infected=["M_I", "B_E", "B_I"])

print("DFE:", modelo.dfe)
print("F:", modelo.new_infections)
print("V:", modelo.transitions)
modelo.R0

DFE: {C: Lambda_C/(alpha + mu_C), M_U: Lambda_C*alpha/(mu_MU*(alpha + mu_C)), M_I: 0, B_E: 0, B_I: 0}
F: {M_I: B_E*M_U*epsilon*nu, B_E: 0, B_I: B_E*M_U*epsilon}
V: {M_I: C*M_I*lambda + M_I*mu_MI + M_I*rho, B_E: B_E*C*eta + B_E*mu_BE - M_I*r*rho, B_I: -B_I*M_I*mu*sigma/(K_m + M_I*sigma) + B_I*M_U*theta + B_I*mu_BI}


Lambda_C*alpha*epsilon*nu*r*rho*(alpha + mu_C)/(mu_MU*(Lambda_C*eta + alpha*mu_BE + mu_BE*mu_C)*(Lambda_C*lambda + alpha*mu_MI + alpha*rho + mu_C*mu_MI + mu_C*rho))

In [27]:
from pyr0compute import R0Model

modelo = R0Model("""
    dE/dt = g_E - (d_E + tau_E*V/(V + zeta_VE))*E
    dI/dt = tau_E*V*E/(V + zeta_VE) - (d_I + tau_I*C_0/(C_0 + zeta_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B_0/(B_0 + zeta_BV))*V
""", infected=["I", "V"])

modelo.R0

g_E*nu*tau_E*(B_0 + zeta_BV)*(C_0 + zeta_CI)/(d_E*zeta_VE*(B_0*d_V + B_0*tau_V + d_V*zeta_BV)*(C_0*d_I + C_0*tau_I + d_I*zeta_CI))

In [28]:
hv = R0Model("""
    dE/dt = g_E - (d_E + tau_E*V/(V + zeta_VE))*E
    dI/dt = tau_E*V*E/(V + zeta_VE) - (d_I + tau_I*C/(C + zeta_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + zeta_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + zeta_IH) + rho_H*R/(R + zeta_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + zeta_HC) + rho_C*R/(R + zeta_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + zeta_HB) + rho_B*R/(R + zeta_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + zeta_HR) + rho_R)*R
""", infected=["I", "V"])

hv.dfe

{E: g_E/d_E,
 I: 0,
 V: 0,
 H: (-d_H*g_R*zeta_HR + d_R*g_H*zeta_RH + g_H*rho_R*zeta_RH - g_H*tau_R*zeta_RH - g_R*rho_H*zeta_HR + (-(d_H*d_R*zeta_HR*zeta_RH - d_H*g_R*zeta_HR + d_H*rho_R*zeta_HR*zeta_RH + d_R*g_H*zeta_RH - g_H*g_R + g_H*rho_R*zeta_RH - g_H*tau_R*zeta_RH - g_R*rho_H*zeta_HR)/(2*(d_H*d_R*zeta_HR + d_H*rho_R*zeta_HR + d_R*g_H + d_R*rho_H*zeta_HR + g_H*rho_R - g_H*tau_R + rho_H*rho_R*zeta_HR)) + sqrt(d_H**2*d_R**2*zeta_HR**2*zeta_RH**2 + 2*d_H**2*d_R*g_R*zeta_HR**2*zeta_RH + 2*d_H**2*d_R*rho_R*zeta_HR**2*zeta_RH**2 + d_H**2*g_R**2*zeta_HR**2 + 2*d_H**2*g_R*rho_R*zeta_HR**2*zeta_RH + d_H**2*rho_R**2*zeta_HR**2*zeta_RH**2 + 2*d_H*d_R**2*g_H*zeta_HR*zeta_RH**2 + 4*d_H*d_R*g_H*g_R*zeta_HR*zeta_RH + 4*d_H*d_R*g_H*rho_R*zeta_HR*zeta_RH**2 - 2*d_H*d_R*g_H*tau_R*zeta_HR*zeta_RH**2 + 2*d_H*d_R*g_R*rho_H*zeta_HR**2*zeta_RH + 2*d_H*g_H*g_R**2*zeta_HR + 4*d_H*g_H*g_R*rho_R*zeta_HR*zeta_RH - 2*d_H*g_H*g_R*tau_R*zeta_HR*zeta_RH + 2*d_H*g_H*rho_R**2*zeta_HR*zeta_RH**2 - 2*d_H*g_H*rho_R*ta